# Behavior-based recommender (Collaborative Filtering)

# Importing Libraries

In [7]:
import pandas as pd
import json

# Converting Json file to csv

In [2]:
file_path = r'D:\JUPYTER NOTEBOOK\ML PROJECT\Recommendation System\Clothing_Shoes_and_Jewelry.json'
output_csv = 'reviews_full.csv'

chunk_size = 100000   
max_rows = 10000000  

data = []
first = True

with open(file_path, 'r') as f:
    for i, line in enumerate(f):

        if i >= max_rows:
            break

        obj = json.loads(line)

        data.append({
            'reviewerID': obj.get('reviewerID'),
            'asin': obj.get('asin'),
            'overall': obj.get('overall')
        })

        if len(data) == chunk_size:
            df = pd.DataFrame.from_records(data)  
            df.to_csv(output_csv, mode='a', index=False, header=first)

            first = False
            data.clear()   

            print(f"Processed {i} rows")

    # last chunk
    if data:
        df = pd.DataFrame.from_records(data)
        df.to_csv(output_csv, mode='a', index=False, header=first)

print("✅ Done (FAST + limited to 1 crore rows)")

Processed 99999 rows
Processed 199999 rows
Processed 299999 rows
Processed 399999 rows
Processed 499999 rows
Processed 599999 rows
Processed 699999 rows
Processed 799999 rows
Processed 899999 rows
Processed 999999 rows
Processed 1099999 rows
Processed 1199999 rows
Processed 1299999 rows
Processed 1399999 rows
Processed 1499999 rows
Processed 1599999 rows
Processed 1699999 rows
Processed 1799999 rows
Processed 1899999 rows
Processed 1999999 rows
Processed 2099999 rows
Processed 2199999 rows
Processed 2299999 rows
Processed 2399999 rows
Processed 2499999 rows
Processed 2599999 rows
Processed 2699999 rows
Processed 2799999 rows
Processed 2899999 rows
Processed 2999999 rows
Processed 3099999 rows
Processed 3199999 rows
Processed 3299999 rows
Processed 3399999 rows
Processed 3499999 rows
Processed 3599999 rows
Processed 3699999 rows
Processed 3799999 rows
Processed 3899999 rows
Processed 3999999 rows
Processed 4099999 rows
Processed 4199999 rows
Processed 4299999 rows
Processed 4399999 rows

# User Filtering

In [3]:
# Keeping only users who interacted enough

chunk_size = 200000
user_counts = {}

# Count user interactions
for chunk in pd.read_csv("reviews_full.csv", chunksize=chunk_size):
    counts = chunk['reviewerID'].value_counts()

    # Combining counts from all chunks
    
    for user, count in counts.items():
        user_counts[user] = user_counts.get(user, 0) + count

# Keep strong users
valid_users = {user for user, count in user_counts.items() if count >= 5}

# Filter and save
first = True

for chunk in pd.read_csv("reviews_full.csv", chunksize=chunk_size):
    chunk = chunk[chunk['reviewerID'].isin(valid_users)]
    
    chunk.to_csv("filtered_reviews.csv", mode='a', index=False, header=first)
    first = False

print("user filtering done")

user filtering done


# item Filtering

In [4]:
# Keeping only products with enough interactions with users

item_counts = {}

for chunk in pd.read_csv("filtered_reviews.csv", chunksize=200000):
    counts = chunk['asin'].value_counts()
    
    for item, count in counts.items():
        item_counts[item] = item_counts.get(item, 0) + count

valid_items = {item for item, count in item_counts.items() if count >= 5}

# Apply filter
first = True

for chunk in pd.read_csv("filtered_reviews.csv", chunksize=200000):
    chunk = chunk[chunk['asin'].isin(valid_items)]
    
    chunk.to_csv("final_reviews.csv", mode='a', index=False, header=first)
    first = False

print("Item filtering done")

Item filtering done


**1. Reduction in Dataset Size**
- After applying user and item filtering, the dataset size reduced significantly
- Low-quality data (users with very few interactions and rarely rated products) was removed
- Final dataset became more compact and meaningful


**2. Increase in Interaction Density**
- Before filtering → sparse data (many missing interactions)
- After filtering → denser user-item interactions

 This means:

- More overlap between users and products
- Better relational patterns

**3. Removal of Noise**
- Users with only 1–2 interactions were removed
- Products with very few ratings were excluded

- Reduced randomness
- Improved signal quality

**Filtering users and items based on interaction thresholds significantly improves the quality of the dataset. By retaining only active users and frequently interacted products, the dataset becomes denser and more informative. This enhances the performance of collaborative filtering models by improving similarity calculations, reducing sparsity, and eliminating noise. As a result, the recommendation system can generate more accurate, reliable, and meaningful recommendations.**

In [8]:
# Checking size 
df_reviews = pd.read_csv("final_reviews.csv")

print(df_reviews.shape)

(3082085, 3)


In [9]:
df_reviews.head(1)

,reviewerID,asin,overall
0,A2IC3NZN488KWK,0871167042,5.0


In [10]:
# Extracting valid asins
valid_asins = set(df_reviews['asin'].unique())

print('Total products :',len(valid_asins))

Total products : 58203


# Loading Metadata

In [11]:
import json
import pandas as pd

file_path = r'D:\JUPYTER NOTEBOOK\ML PROJECT\Recommendation System\meta_Clothing_Shoes_and_Jewelry.json'

data = []

with open(file_path, 'r') as f:
    for i, line in enumerate(f):

        obj = json.loads(line)
        asin = obj.get('asin')

        #  Only load useful products
        if asin in valid_asins:
            data.append({
                'asin': asin,
                'title': obj.get('title'),
                'brand': obj.get('brand'),
                'category': obj.get('category'),
                'feature': obj.get('feature'),
                'also_view': obj.get('also_view'),
                'also_buy': obj.get('also_buy'),
                'imageURLHighRes': obj.get('imageURLHighRes')
            })

        if i % 100000 == 0:
            print(f"Processed {i} rows")

df_meta = pd.DataFrame.from_records(data)

print("Metadata loaded:", df_meta.shape)

Processed 0 rows
Processed 100000 rows
Processed 200000 rows
Processed 300000 rows
Processed 400000 rows
Processed 500000 rows
Processed 600000 rows
Processed 700000 rows
Processed 800000 rows
Processed 900000 rows
Processed 1000000 rows
Processed 1100000 rows
Processed 1200000 rows
Processed 1300000 rows
Processed 1400000 rows
Processed 1500000 rows
Processed 1600000 rows
Processed 1700000 rows
Processed 1800000 rows
Processed 1900000 rows
Processed 2000000 rows
Processed 2100000 rows
Processed 2200000 rows
Processed 2300000 rows
Processed 2400000 rows
Processed 2500000 rows
Processed 2600000 rows
Metadata loaded: (59027, 8)


In [12]:
# checking missing values in df_meta
df_meta.isna().sum()

asin                   0
title                  0
brand              24354
category               0
feature              878
also_view          13917
also_buy           24995
imageURLHighRes     8104
dtype: int64

In [13]:
# remove missing important values
df_meta = df_meta.dropna(subset=['asin', 'title'])

print("After cleaning:", df_meta.shape)

After cleaning: (59027, 8)


In [14]:
df_meta.sample(1)

,asin,title,brand,category,feature,also_view,also_buy,imageURLHighRes
2287,B0009BFWU8,Giorgio Brutini Men's Cortland Oxford,Giorgio Brutini,"[Clothing, Shoes & Jewelry, Men, Shoes, Oxfords]","[Clothing, Shoes & Jewelry, Men, Shoes, Oxford...","[B000HGI07W, B000HGK1VA, B00LMWU4ME, B01N5VGYI...","[B000HGI07W, B000HGK1VA, B01N5VGYI6, B00LMWU4M...",[https://images-na.ssl-images-amazon.com/image...


In [15]:
df_meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59027 entries, 0 to 59026
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   asin             59027 non-null  object
 1   title            59027 non-null  object
 2   brand            34673 non-null  object
 3   category         59027 non-null  object
 4   feature          58149 non-null  object
 5   also_view        45110 non-null  object
 6   also_buy         34032 non-null  object
 7   imageURLHighRes  50923 non-null  object
dtypes: object(8)
memory usage: 3.6+ MB


In [16]:
print(type(df_meta['also_buy'].iloc[0]))

<class 'list'>


In [17]:
# image → take first image
def get_image(x):
    if isinstance(x, list) and len(x) > 0:
        return x[0]
    return None

df_meta['image'] = df_meta['imageURLHighRes'].apply(get_image)


# category → list to string
df_meta['category'] = df_meta['category'].apply(
    lambda x: ' > '.join(x) if isinstance(x, list) else None
)

# feature → list to string
df_meta['feature'] = df_meta['feature'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) else None
)

In [18]:
df_meta = df_meta[['asin', 'title', 'brand', 'category',
                   'feature', 'also_view', 'also_buy', 'image']]

print("Final metadata ready")


Final metadata ready


In [19]:

df_meta.head()

,asin,title,brand,category,feature,also_view,also_buy,image
0,0871167042,Organic Wire and Metal Jewelry: Stunning Piece...,Visit Amazon's Eva M. Sherman Page,"Clothing, Shoes & Jewelry > Women > Jewelry",None,"[1634509374, 1612433030, 1627000763, 163250347...","[1627002332, 1634509374, 1627000763, 161243303...",None
1,1519588135,One White Lie (The Barrington Billionaire's Se...,Visit Amazon's Jeannette Winters Page,"Clothing, Shoes & Jewelry > Women > Clothing",None,"[1984152041, 1723965111]","[1535298278, 1974248674, 1984152041, 099924253...",None
2,1579652956,Itty-Bitty Hats: cute and cuddly caps to knit ...,Visit Amazon's Susan B. Anderson Page,"Clothing, Shoes & Jewelry > Men > Accessories",None,"[0823099032, 1612124801, 1579653766, 157965334...","[1579653766, 1579653340, 0823099032, 160900215...",None
3,1936023857,Carson Dellosa Kids Name Tags (150003),Carson-Dellosa,"Clothing, Shoes & Jewelry > Luggage & Travel G...",40 self-adhesive name tags per pack Available ...,"[B01GFTI4QQ, B008G0WGDU, 1594419019, B00D5T3PF...","[B00D5T3PF0, 193602392X, B002IXIWRA, B002IXEMG...",None
4,3979050432,Face Ski Mask 3 Hole (More Colors)- Black,TopHeadwear,"Clothing, Shoes & Jewelry > Men > Accessories ...",Package Dimensions:\n \n8.8...,"[B07C9WXDBY, B077Q7PHFS, B075Y58DVV]",None,https://images-na.ssl-images-amazon.com/images...


In [20]:
df_meta.isna().sum()

asin             0
title            0
brand        24354
category         0
feature        878
also_view    13917
also_buy     24995
image         8104
dtype: int64

In [21]:
df_meta['brand'] = df_meta['brand'].fillna("Unknown")
df_meta['feature'] = df_meta['feature'].fillna("")
df_meta['image'] = df_meta['image'].fillna("https://via.placeholder.com/150")

In [22]:
df_meta.isna().sum()

asin             0
title            0
brand            0
category         0
feature          0
also_view    13917
also_buy     24995
image            0
dtype: int64

In [23]:
df_final = pd.merge(df_reviews, df_meta, on='asin', how='inner')

In [24]:
df_final.shape

(3348043, 10)

In [25]:
df_final.sample(1)

,reviewerID,asin,overall,title,brand,category,feature,also_view,also_buy,image
1679910,A24ITMOWMNCE1L,B001A3ZF7Y,5.0,Dockers Men's Jean Cut Stretch Straight Fit Pant,Unknown,"Clothing, Shoes & Jewelry > Men > Clothing","98% Cotton, 2% Elastane Imported Button closur...","[B077PQHH9L, B01BNSKFHU, B0779P635Z, B0748RQ2F...","[B0779P635Z, B077PQHH9L, B0779MYSDF, B077PNMM5...",https://images-na.ssl-images-amazon.com/images...


In [26]:
df_final.isna().sum()

reviewerID         0
asin               0
overall            0
title              0
brand              0
category           0
feature            0
also_view     222435
also_buy      513575
image              0
dtype: int64

In [27]:
df_final.duplicated(subset=['reviewerID', 'asin']).sum()

867075

In [28]:
df_final[df_final.duplicated(subset=['reviewerID', 'asin'], keep=False)].head(5)

,reviewerID,asin,overall,title,brand,category,feature,also_view,also_buy,image
0,A2IC3NZN488KWK,0871167042,5.0,Organic Wire and Metal Jewelry: Stunning Piece...,Visit Amazon's Eva M. Sherman Page,"Clothing, Shoes & Jewelry > Women > Jewelry",,"[1634509374, 1612433030, 1627000763, 163250347...","[1627002332, 1634509374, 1627000763, 161243303...",https://via.placeholder.com/150
1,A2G9GWQEWWNQUB,0871167042,5.0,Organic Wire and Metal Jewelry: Stunning Piece...,Visit Amazon's Eva M. Sherman Page,"Clothing, Shoes & Jewelry > Women > Jewelry",,"[1634509374, 1612433030, 1627000763, 163250347...","[1627002332, 1634509374, 1627000763, 161243303...",https://via.placeholder.com/150
2,A3NI5OGW35SLY2,0871167042,5.0,Organic Wire and Metal Jewelry: Stunning Piece...,Visit Amazon's Eva M. Sherman Page,"Clothing, Shoes & Jewelry > Women > Jewelry",,"[1634509374, 1612433030, 1627000763, 163250347...","[1627002332, 1634509374, 1627000763, 161243303...",https://via.placeholder.com/150
3,A3M6UXIK7XTA7A,0871167042,4.0,Organic Wire and Metal Jewelry: Stunning Piece...,Visit Amazon's Eva M. Sherman Page,"Clothing, Shoes & Jewelry > Women > Jewelry",,"[1634509374, 1612433030, 1627000763, 163250347...","[1627002332, 1634509374, 1627000763, 161243303...",https://via.placeholder.com/150
4,A22ZX01TPWQY4G,0871167042,2.0,Organic Wire and Metal Jewelry: Stunning Piece...,Visit Amazon's Eva M. Sherman Page,"Clothing, Shoes & Jewelry > Women > Jewelry",,"[1634509374, 1612433030, 1627000763, 163250347...","[1627002332, 1634509374, 1627000763, 161243303...",https://via.placeholder.com/150


In [29]:
# Handling duplicates
df_final = df_final.groupby(['reviewerID', 'asin'], as_index=False).agg({
    'overall': 'mean',
    'title': 'first',
    'brand': 'first',
    'category': 'first',
    'feature': 'first',
    'also_view': 'first',
    'also_buy': 'first',
    'image': 'first'
})

In [30]:
# Checking missing values
df_final.duplicated(subset=['reviewerID', 'asin']).sum()

0

In [31]:
df_final.sample(1)

,reviewerID,asin,overall,title,brand,category,feature,also_view,also_buy,image
71165,A13WLANULBQV7E,B004FE2JTM,5.0,TrendsBlue Classic Soft Unisex Solid Color War...,TrendsBlue,"Clothing, Shoes & Jewelry > Women > Accessorie...","100% Premium soft acrylic Fringe style: 74"" x ...","[B07BV59PKW, B01MG271Q2, B016WRFKCW, B077YYLX1...",[B07HQJZRCF],https://images-na.ssl-images-amazon.com/images...


# ML-starts


## Content-based-model

In [39]:
# selecting rquired columns
df_cb = df_final[['asin', 'title', 'brand', 'category', 'feature']].copy()


In [40]:
df_cb.shape

(2480968, 5)

In [41]:
df_cb.head()

,asin,title,brand,category,feature
0,B0002IC652,Mirrored Aviators Silver Metal Aviator Sunglas...,Unknown,"Clothing, Shoes & Jewelry > Women > Accessorie...","Clothing, Shoes & Jewelry Women Accessories Su..."
1,B0012G297S,Skechers Sport Women's D'Lites Original Non-Me...,Unknown,"Clothing, Shoes & Jewelry > Women > Shoes > At...",100% Leather Imported Rubber sole Bold sneaker...
2,B00ADBMIN8,925 Sterling Silver 0.06 tcw Basket Setting 2M...,Metal Factory,"Clothing, Shoes & Jewelry > Women > Jewelry > ...",Grade AAAAA Quality Cubic Zirconia Rhodium pla...
3,B001LRNGZC,Carhartt Flame-Resistant Twill Shirt with Pock...,Carhartt,"Clothing, Shoes & Jewelry > Men > Clothing > S...","7 oz. lightweight, flame-resistant twill Butto..."
4,B001LRR9IM,var aPageStart = (new Date()).getTime();\nvar ...,Carhartt,"Clothing, Shoes & Jewelry > Men > Clothing > S...","7 oz. lightweight, flame-resistant twill Butto..."


In [42]:
# Removing duplicates
df_cb = df_cb.drop_duplicates('asin')

In [43]:
# keeping only popular products 
product_counts = df_final['asin'].value_counts()

# Keep products with at least 5 interactions
valid_products = product_counts[product_counts >= 5].index

df_cb = df_cb[df_cb['asin'].isin(valid_products)]

In [44]:
df_cb.shape

(57097, 5)

In [45]:
# extracting unique image per product
df_image = df_final[['asin', 'image']].drop_duplicates('asin')

# merge with df_cb
df_cb = pd.merge(df_cb, df_image, on='asin', how='left')

In [46]:
df_cb['image'].isnull().sum()

0

In [47]:
# filling missing values 
df_cb = df_cb.fillna('')
df_cb = df_cb.drop_duplicates(subset='title')
# creating combinex text
df_cb['combined'] = (
    df_cb['title'] + ' ' +
    df_cb['brand'] + ' ' +
    df_cb['category'] + ' ' +
    df_cb['feature']
)

df_cb = df_cb[['asin','combined']]

In [48]:
df_cb.shape

(53941, 2)

In [56]:
df_cb.head()

,asin,combined
0,B0002IC652,mirror aviat silver metal aviat sunglass silve...
1,B0012G297S,skecher sport women s d lite origin non memori...
2,B00ADBMIN8,sterl silver tcw basket set mm clear round cz ...
3,B001LRNGZC,carhartt flame resist twill shirt with pocket ...
4,B001LRR9IM,var apagestart new date gettim var ue t ue t n...


In [79]:
from nltk.stem.porter import PorterStemmer
import re

ps = PorterStemmer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub('[^a-zA-Z]', ' ', text)
    words = text.split()
    
    words = [ps.stem(word) for word in words if len(word) > 2]  # remove small words
    
    return " ".join(words)

In [80]:
df_cb['combined'] = df_cb['combined'].apply(preprocess)

In [81]:
# bag of words 
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(
    max_features=5000,          
    stop_words='english',
    ngram_range=(1,2)           
)


In [82]:
# creating vectors
vectors = cv.fit_transform(df_cb['combined'])

# creating indices mapping
indices = pd.Series(df_cb.index, index=df_cb['asin']).drop_duplicates()

In [84]:
import pickle

pickle.dump(cv, open("cv.pkl", "wb"))
pickle.dump(vectors, open("vectors.pkl", "wb"))
pickle.dump(indices, open("indices.pkl", "wb"))
df_cb.to_pickle("df_cb.pkl")

In [85]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_similar_products(asin, top_n=5):

    if asin not in indices:
        print("Example ASIN:", df_cb['asin'].iloc[0])
        return "Product not found"

    idx = indices[asin]

    sim_scores = cosine_similarity(
        vectors[idx],
        vectors
    ).flatten()

    sim_indices = sim_scores.argsort()[-top_n-1:-1][::-1]

    return df_cb.iloc[sim_indices][['asin', 'title']]

In [76]:
vectors.shape

(53941, 3000)

In [77]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [ ]:
cv.get_feature_names_out()

array(['aa', 'aaa', 'ab', ..., 'zoom', 'zwl', 'zzvaml'], dtype=object)

In [ ]:
len(cv.get_feature_names_out())

3000

In [ ]:
df_cb['combined'][0]

'mirror aviat silver metal aviat sunglass silver unknown cloth shoe jewelri women accessori sunglass eyewear accessori sunglass cloth shoe jewelri women accessori sunglass eyewear accessori sunglass you can return thi item for ani reason and get a full refund no ship charg the item must be return in new and unus condit read the full return polici go to your order to start the return print the return ship label ship it'

In [ ]:
preprocess('mirror aviat silver metal aviat sunglass silver unknown cloth shoe jewelri women accessori sunglass eyewear accessori sunglass cloth shoe jewelri women accessori sunglass eyewear accessori sunglass you can return thi item for ani reason and get a full refund no ship charg the item must be return in new and unus condit read the full return polici go to your order to start the return print the return ship label ship it')

'mirror aviat silver metal aviat sunglass silver unknown cloth shoe jewelri women accessori sunglass eyewear accessori sunglass cloth shoe jewelri women accessori sunglass eyewear accessori sunglass you can return thi item for ani reason and get a full refund no ship charg the item must be return in new and unu condit read the full return polici go to your order to start the return print the return ship label ship it'

In [87]:
df_cb.head(1)

,asin,combined
0,B0002IC652,mirror aviat silver metal aviat sunglass silve...
